## Uvod 

Ta lekcija bo pokrivala: 
- Kaj je klic funkcije in njene uporabe 
- Kako ustvariti klic funkcije z uporabo Azure OpenAI 
- Kako integrirati klic funkcije v aplikacijo 

## Cilji učenja 

Po končani tej lekciji boste znali in razumeli: 

- Namen uporabe klicev funkcij 
- Nastavitev klica funkcije z uporabo storitve Azure Open AI 
- Oblikovanje učinkovitih klicev funkcij za primer uporabe vaše aplikacije 


## Razumevanje klicev funkcij 

Za to lekcijo želimo zgraditi funkcijo za naš startup na področju izobraževanja, ki uporabnikom omogoča uporabo klepetalnega robota za iskanje tehničnih tečajev. Priporočali bomo tečaje, ki ustrezajo njihovi ravni znanja, trenutni vlogi in tehnologiji, ki jih zanima. 

Za dokončanje bomo uporabili kombinacijo: 
 - `Azure Open AI` za ustvarjanje klepeta za uporabnika
 - `Microsoft Learn Catalog API`, da pomagamo uporabnikom najti tečaje na podlagi zahtev uporabnika 
 - `Klic funkcij` za prevzem uporabnikove poizvedbe in pošiljanje funkciji, ki izvede API zahtevo. 

Za začetek si poglejmo, zakaj bi želeli uporabiti klic funkcij: 


### Zakaj klic funkcije 

Če ste opravili katero koli drugo lekcijo v tem tečaju, verjetno že razumete moč uporabe velikih jezikovnih modelov (LLM-jev). Upamo, da lahko vidite tudi nekatere njihove omejitve. 

Klic funkcije je funkcija storitve Azure Open AI, ki omogoča premagovanje naslednjih omejitev: 
1) Dosleden format odgovora 
2) Zmožnost uporabe podatkov iz drugih virov aplikacije v kontekstu klepeta 

Pred klicem funkcije so bili odgovori LLM-jev nestrukturirani in nedosledni. Razvijalci so morali pisati zahtevno kodo za preverjanje, da so lahko obravnavali vsako različico odgovora. 

Uporabniki niso mogli dobiti odgovorov, kot je "Kakšno je trenutno vreme v Stockholmu?". To je zato, ker so bili modeli omejeni na čas, ko so bili podatki usposobljeni. 

Poglejmo spodnji primer, ki prikazuje ta problem: 

Recimo, da želimo ustvariti bazo podatkov o študentih, da jim lahko predlagamo pravi tečaj. Spodaj imamo dva opisa študentov, ki sta si zelo podobna v podatkih, ki jih vsebujeta.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Radi to pošljemo LLM, da obdela podatke. To lahko kasneje uporabimo v naši aplikaciji za pošiljanje na API ali shranjevanje v bazo podatkov. 

Ustvarimo dva enaka poziva, v katerih LLM navodimo, katere informacije nas zanimajo: 


Želimo to poslati LLM-ju, da razčleni dele, ki so pomembni za naš izdelek. Tako lahko ustvarimo dva enaka poziva za usmerjanje LLM-ja: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Po ustvarjanju teh dveh pozivov jih bomo poslali LLM-ju z uporabo `client.responses.create`. Poziv shranimo v spremenljivko `input` in vlogi dodelimo `user`. To je za posnemanje sporočila uporabnika, ki je napisano za klepetalni robot.



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Zdaj lahko pošljemo oba zahtevka LLM-ju in preučimo odgovor, ki ga prejmemo. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Čeprav so pozivi enaki in so opisi podobni, lahko dobimo različne formate lastnosti `Grades`. 

Če zgornjo celico zaženete večkrat, je format lahko `3.7` ali `3.7 GPA`. 

To je zato, ker LLM sprejme nestrukturirane podatke v obliki zapisanega poziva in vrne prav tako nestrukturirane podatke. Potrebujemo strukturiran format, da vemo, kaj pričakovati pri shranjevanju ali uporabi teh podatkov.

Z uporabo funkcijskega klica lahko zagotovimo, da dobimo nazaj strukturirane podatke. Pri uporabi funkcijskega klica LLM dejansko ne kliče ali izvaja nobenih funkcij. Namesto tega ustvarimo strukturo, ki naj jo LLM upošteva pri odgovorih. Nato uporabimo te strukturirane odgovore, da vemo, katero funkcijo naj izvedemo v naših aplikacijah.  
 


![Diagram poteka klica funkcije](../../../../translated_images/sl/Function-Flow.083875364af4f4bb.webp)


Nato lahko vzamemo tisto, kar funkcija vrne, in to pošljemo nazaj LLM-ju. LLM bo nato odgovarjal v naravnem jeziku, da odgovori na uporabnikovo poizvedbo. 


### Primeri uporabe klicev funkcij

**Klicanje zunanjih orodij**
Chatboti so odlični pri zagotavljanju odgovorov na vprašanja uporabnikov. Z uporabo klicev funkcij lahko chatboti uporabniška sporočila uporabijo za izvedbo določenih opravil. Na primer, študent lahko zaprosi chatbota: "Pošlji e-pošto mojemu inštruktorju z obvestilom, da potrebujem več pomoči pri tem predmetu." To lahko sproži klic funkcije `send_email(to: string, body: string)`


**Ustvarjanje API ali poizvedb v zbirki podatkov**
Uporabniki lahko najdejo informacije z uporabo naravnega jezika, ki se pretvori v oblikovano poizvedbo ali API zahtevek. Primer tega je učitelj, ki vpraša: "Kateri študenti so dokončali zadnjo nalogo," kar lahko sproži klic funkcije z imenom `get_completed(student_name: string, assignment: int, current_status: string)`


**Ustvarjanje strukturiranih podatkov**
Uporabniki lahko vzamejo blok besedila ali CSV in uporabijo LLM za izvleček pomembnih informacij iz njega. Na primer, študent lahko spremeni Wikipedijin članek o mirovnih sporazumih v AI kartice za učenje. To je mogoče storiti z uporabo funkcije `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Ustvarjanje vašega prvega klica funkcije 

Postopek ustvarjanja klica funkcije vključuje 3 glavne korake: 
1. Klic API-ja Chat Completions z seznamom vaših funkcij in uporabniškim sporočilom 
2. Preberite odgovor modela, da izvedete akcijo, npr. izvedete funkcijo ali kličete API 
3. Naredite še en klic API-ja Chat Completions z odgovorom vaše funkcije, da uporabite te informacije za ustvarjanje odgovora uporabniku. 


![Tok klica funkcije](../../../../translated_images/sl/LLM-Flow.3285ed8caf4796d7.webp)


### Elementi klica funkcije 

#### Vhod uporabnika 

Prvi korak je ustvariti uporabnikovo sporočilo. To lahko dinamično dodelite tako, da vzamete vrednost tekstovnega vnosa, ali pa tukaj dodelite vrednost. Če je to vaš prvič, da delate z API-jem Chat Completions, moramo določiti `role` in `content` sporočila. 

`role` je lahko `system` (določanje pravil), `assistant` (model) ali `user` (končni uporabnik). Za klic funkcije bomo to določili kot `user` in podali primer vprašanja. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Ustvarjanje funkcij. 

Naslednji korak je definirati funkcijo in parametre te funkcije. Tukaj bomo uporabili samo eno funkcijo z imenom `search_courses`, vendar lahko ustvarite več funkcij.

**Pomembno** : Funkcije so vključene v sistemsko sporočilo za LLM in so vključene v količino razpoložljivih žetonov, ki jih imate na voljo. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definicije** 

`name` -  Ime funkcije, ki jo želimo poklicati. 

`description` - To je opis, kako funkcija deluje. Pomembno je biti specifičen in jasen 

`parameters` - Seznam vrednosti in format, ki jih želite, da model ustvari v svojem odgovoru 


`type` -  Podatkovna vrsta, v kateri bodo shranjene lastnosti 

`properties` - Seznam specifičnih vrednosti, ki jih bo model uporabil za svoj odgovor 


`name` - ime lastnosti, ki jo bo model uporabil v svojem formatiranem odgovoru 

`type` - Podatkovna vrsta te lastnosti 

`description` - Opis specifične lastnosti 


**Izbirno**

`required` - zahtevana lastnost za dokončanje klica funkcije 


### Klic funkcije
Po definiranju funkcije jo moramo sedaj vključiti v klic API-ja za popravke klepeta. To naredimo tako, da v zahtevo dodamo `functions`. V tem primeru `functions=functions`.

Obstaja tudi možnost nastavitve `function_call` na `auto`. To pomeni, da dovolimo LLM-ju, da sam odloči, katera funkcija naj se pokliče na podlagi uporabnikovega sporočila, namesto da bi jo določili mi sami.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Oglejmo si zdaj odgovor in poglejmo, kako je oblikovan: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Vidite lahko, da je bila funkcija poimenovana in glede na uporabnikovo sporočilo je LLM uspel najti podatke, ki ustrezajo argumentom funkcije. 


## 3. Integracija funkcijskih klicev v aplikacijo. 


Ko smo preizkusili oblikovani odgovor iz LLM, lahko to zdaj integriramo v aplikacijo. 

### Upravljanje toka 

Za integracijo tega v našo aplikacijo izvedimo naslednje korake: 

Najprej izvedimo klic storitve Open AI in sporočilo shranimo v spremenljivko z imenom `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Zdaj bomo definirali funkcijo, ki bo poklicala Microsoft Learn API, da pridobi seznam tečajev: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Kot najboljšo prakso bomo nato preverili, ali model želi poklicati funkcijo. Nato bomo ustvarili eno izmed razpoložljivih funkcij in jo ujemali s funkcijo, ki se kliče. 
Nato bomo vzeli argumente funkcije in jih preslikali na argumente iz LLM.

Nazadnje bomo dodali sporočilo o klicu funkcije in vrednosti, ki jih je vrnilo sporočilo `search_courses`. To daje LLM vse informacije, ki jih potrebuje
za odgovor uporabniku v naravnem jeziku. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Zdaj bomo poslali posodobljeno sporočilo LLM, da lahko prejmemo odgovor v naravnem jeziku namesto v obliki API JSON. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Koda Izziv

Odlično delo! Za nadaljevanje učenja o Azure Open AI Function Calling lahko zgradite: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst
 - Več parametrov funkcije, ki bi lahko pomagali učencem najti več tečajev. Na voljo API parametre lahko najdete tukaj:
 - Ustvarite še en klic funkcije, ki od udeleženca zahteva več informacij, kot je njihov materni jezik
 - Ustvarite obravnavo napak, kadar klic funkcije in/ali klic API ne vrne primernih tečajev


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
